# Raceline viewer

Any line, any track, interactive: hover a point on the map and it says what the
plan demands there (speed, curvature, lateral demand, width to each wall, body
margin). The same figure is written out as a standalone web page in which the
cursor is linked: hover anywhere and the map point and the s-position on both
profiles follow.

**Kernel:** `.venv-rl` (needs `plotly`, installed there on 2026-09-08). Set
`TRACK` and `LINE` in the first cell and run all.

The map loader, the physics and the track registry are imported from the
pipeline rather than repeated, so the widths, the body margin and the zones are
the optimizer's own.

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO = Path.home() / "Documents/roboracer"
sys.path.insert(0, str(REPO / "raceline"))
sys.path.insert(0, str(REPO / "devkit_ws/src/racer_common"))
from optimize_raceline import TrackMap, map_base, PHYS   # the pipeline's map loader and physics
from racer_common import frames                          # the track registry; plain Python, no ROS needed

TRACK = "icra2026"
LINE = "raceline_a7.0zv_hard_l6.0_corners_h.csv"   # run 23, the ICRA submission
# LINE = frames.TRACKS[TRACK]["raceline"]          # whatever track:=icra2026 launches right now
# LINE = "raceline_a7.0zv_hard.csv"                # run 17, the flag-free fallback
# LINE = "centerline_full.csv"                     # no speed column: geometry only

SHOW_ZONES = True        # shade the registry's lat/v zones on the profiles, ring the margin zones on the map
EMBED_PLOTLY_JS = False  # True: the page opens offline (+4.5 MB); False: it loads plotly.js from the CDN

LINE_DIR = REPO / "raceline" / TRACK
# track_solid when the track has one (walls sealed), else track_clean: the optimizer's rule
tm = TrackMap(map_base(TRACK))
print(f"map {map_base(TRACK).name}: {tm.W} x {tm.H} cells @ {tm.res} m")

## 1. Load a line

Columns are the pipeline's: `s, x, y, psi, kappa, w_r, w_l[, v]`. Everything
derived here is what the plan *asks for*, not what the car did: lateral demand
is `v²|κ|`, body margin is clearance to the nearest wall cell minus half the car
width, and the lap time is segment length over segment mean speed, the
optimizer's "on paper" number. Measured laps live in `VEHICLE_MODEL.md` §7.

In [ ]:
def load_line(path, tm):
    D = np.loadtxt(path, delimiter=",")
    d = dict(name=Path(path).stem, s=D[:, 0], x=D[:, 1], y=D[:, 2], psi=D[:, 3],
             kappa=D[:, 4], w_r=D[:, 5], w_l=D[:, 6],
             v=D[:, 7] if D.shape[1] > 7 else None)
    xy = np.c_[d["x"], d["y"]]
    d["el"] = np.linalg.norm(np.roll(xy, -1, axis=0) - xy, axis=1)   # closed loop: the last segment returns to s = 0
    d["length"] = float(d["el"].sum())
    d["clearance"] = tm.clearance(d["x"], d["y"])                      # point to the nearest wall/unknown cell
    d["body"] = d["clearance"] - PHYS.width / 2                         # the optimizer's body margin
    m = re.search(r"_a(\d+\.\d+)", d["name"])
    d["rung"] = float(m.group(1)) if m else None                        # the a_lat the profile was built at
    if d["v"] is not None:
        d["a_lat"] = d["v"] ** 2 * np.abs(d["kappa"])
        v_seg = 0.5 * (d["v"] + np.roll(d["v"], -1))
        d["lap"] = float((d["el"] / v_seg).sum())
    return d


def describe(d):
    i = int(np.argmin(d["body"]))
    print(f"{d['name']}: {len(d['s'])} pts, {d['length']:.2f} m, |kappa| max {np.abs(d['kappa']).max():.2f} 1/m, "
          f"tightest body margin {d['body'][i]:+.2f} m at s {d['s'][i]:.1f}")
    if d["v"] is not None:
        j = int(np.argmax(d["a_lat"]))
        rung = f" (rung {d['rung']})" if d["rung"] else ""
        print(f"    {d['lap']:.3f} s on paper, v {d['v'].min():.2f}-{d['v'].max():.2f} m/s, "
              f"lateral demand max {d['a_lat'][j]:.2f} m/s² at s {d['s'][j]:.1f}{rung}")


d = load_line(LINE_DIR / LINE, tm)
describe(d)

## 2. The figure

Left: the occupancy grid with the line coloured by speed. Right: the speed
profile and the lateral demand against the tire's asymptote (4.90, what it
holds at any slip angle) and the rung the profile was built at. With
`SHOW_ZONES` the registry's lateral and speed zones are shaded on the profiles
and the margin zones are ringed on the map, so a hardened corner is visible as
such. Zone s-ranges are the registry's; a few tenths of a metre of slop against
this line's s is normal.

In [ ]:
# customdata columns, identical on every trace: 0 s, 1 x, 2 y, 3 v, 4 kappa, 5 a_lat, 6 w_r, 7 w_l, 8 body.
# The page's linked cursor reads 0-2 (write_page below).
HOVER_LINE = ("<b>s = %{customdata[0]:.2f} m</b><br>"
              "v = %{customdata[3]:.2f} m/s<br>"
              "kappa = %{customdata[4]:+.3f} 1/m<br>"
              "lateral demand = %{customdata[5]:.2f} m/s²<br>"
              "width R %{customdata[6]:.2f} / L %{customdata[7]:.2f} m<br>"
              "body margin = %{customdata[8]:.2f} m")
HOVER_GEOM = ("<b>s = %{customdata[0]:.2f} m</b><br>"
              "kappa = %{customdata[4]:+.3f} 1/m<br>"
              "width R %{customdata[6]:.2f} / L %{customdata[7]:.2f} m<br>"
              "body margin = %{customdata[8]:.2f} m")


def hover_data(d):
    nan = np.full_like(d["s"], np.nan)
    v = d["v"] if d["v"] is not None else nan
    a = d["a_lat"] if d["v"] is not None else nan
    return np.c_[d["s"], d["x"], d["y"], v, d["kappa"], a, d["w_r"], d["w_l"], d["body"]]


def map_trace(tm):
    # 0 wall, 1 unknown, 2 free. PGM row 0 is the top of the image = max y, so flip for a heatmap.
    z = np.where(tm.free, 2, np.where(tm.img == 0, 0, 1))
    return go.Heatmap(z=np.flipud(z), x0=tm.ox + tm.res / 2, dx=tm.res, y0=tm.oy + tm.res / 2, dy=tm.res,
                      zmin=0, zmax=2, showscale=False, hoverinfo="skip",
                      colorscale=[[0, "#1b1b1b"], [1 / 3, "#1b1b1b"], [1 / 3, "#b4b4b4"],
                                  [2 / 3, "#b4b4b4"], [2 / 3, "#ffffff"], [1, "#ffffff"]])


def zones(spec, n):
    """'s0:s1:a,...' -> [(s0, s1, a), ...]; margin zones have four fields: (s0, s1, side, depth)."""
    out = []
    for z in filter(None, spec.split(",")):
        f = z.split(":")
        out.append((float(f[0]), float(f[1]), f[2], float(f[3])) if n == 4
                   else (float(f[0]), float(f[1]), float(f[2])))
    return out


def _frame(subplot_titles):
    """Map on the left spanning both rows, two profiles stacked on the right, sharing s.
    Shapes 0 and 1 are the s-cursors the page moves on hover; they go first so add_vrect cannot renumber them."""
    fig = make_subplots(rows=2, cols=2, specs=[[{"rowspan": 2}, {}], [None, {}]],
                        column_widths=[0.5, 0.5], horizontal_spacing=0.08, vertical_spacing=0.10,
                        subplot_titles=subplot_titles)
    for r in (2, 3):
        fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref=f"x{r}", yref=f"y{r} domain", visible=False,
                      line=dict(color="black", width=1, dash="dot"))
    return fig


def _finish(fig, title, height=820):
    # the cursor the page moves on hover: always the LAST trace (write_page relies on that)
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                             marker=dict(symbol="circle-open", size=16, color="black", line_width=2)), row=1, col=1)
    fig.update_xaxes(title="x [m]", row=1, col=1)
    fig.update_yaxes(title="y [m]", scaleanchor="x", scaleratio=1, row=1, col=1)
    fig.update_xaxes(matches="x2", title="s [m]", row=2, col=2)
    fig.update_yaxes(title="m/s", row=1, col=2)
    fig.update_yaxes(title="m/s²", row=2, col=2)
    fig.add_hline(y=PHYS.a_lat_robust, row=2, col=2, line=dict(color="gray", dash="dash"),
                  annotation_text=f"tire asymptote {PHYS.a_lat_robust:.2f}", annotation_position="bottom right")
    fig.update_layout(title=title, height=height, hovermode="closest", template="plotly_white",
                      legend=dict(font_size=10))
    return fig


def make_figure(d, tm, reg=None):
    has_v = d["v"] is not None
    fig = _frame(("", "speed the plan demands", "lateral demand v²|κ|"))
    fig.add_trace(map_trace(tm), row=1, col=1)
    cd = hover_data(d)
    marker = (dict(size=5, color=d["v"], colorscale="Turbo", cmin=float(d["v"].min()), cmax=float(d["v"].max()),
                   colorbar=dict(title="m/s", orientation="h", x=0.23, xanchor="center", y=-0.12, yanchor="top",
                                 len=0.4, thickness=12))
              if has_v else dict(size=4, color="#00b050"))
    fig.add_trace(go.Scatter(x=d["x"], y=d["y"], mode="markers+lines", name=d["name"], customdata=cd,
                             hovertemplate=(HOVER_LINE if has_v else HOVER_GEOM) + "<extra></extra>",
                             marker=marker, line=dict(width=1, color="rgba(0,0,0,0.35)")), row=1, col=1)
    fig.add_trace(go.Scatter(x=[d["x"][0]], y=[d["y"][0]], mode="markers", name="s = 0", hoverinfo="skip",
                             marker=dict(symbol="diamond", size=10, color="black")), row=1, col=1)
    if reg:
        sx, sy, _ = map(float, reg["spawn"])
        fig.add_trace(go.Scatter(x=[sx], y=[sy], mode="markers", name="spawn", hoverinfo="skip",
                                 marker=dict(symbol="star", size=12, color="magenta")), row=1, col=1)
    if has_v:
        fig.add_trace(go.Scatter(x=d["s"], y=d["v"], mode="lines", name="v", customdata=cd,
                                 line=dict(color="#1f77b4"),
                                 hovertemplate="s %{x:.2f} m: v %{y:.2f} m/s<extra></extra>"), row=1, col=2)
        fig.add_trace(go.Scatter(x=d["s"], y=d["a_lat"], mode="lines", name="v²|κ|", customdata=cd,
                                 line=dict(color="#d62728"),
                                 hovertemplate="s %{x:.2f} m: %{y:.2f} m/s²<extra></extra>"), row=2, col=2)
        if d["rung"]:
            fig.add_hline(y=d["rung"], row=2, col=2, line=dict(color="black", dash="dot"),
                          annotation_text=f"rung {d['rung']}", annotation_position="top right")
    if reg and SHOW_ZONES:
        for s0, s1, a in zones(reg.get("lat_zones", ""), 3):
            fig.add_vrect(x0=s0, x1=s1, row=2, col=2, fillcolor="#d62728", opacity=0.10, line_width=0,
                          annotation_text=f"lat {a:g}", annotation_position="top left", annotation_font_size=10)
        for s0, s1, v in zones(reg.get("v_zones", ""), 3):
            fig.add_vrect(x0=s0, x1=s1, row=1, col=2, fillcolor="#1f77b4", opacity=0.10, line_width=0,
                          annotation_text=f"v {v:g}", annotation_position="top left", annotation_font_size=10)
        for s0, s1, side, depth in zones(reg.get("margin_zones", ""), 4):
            m = (d["s"] >= s0) & (d["s"] <= s1)
            fig.add_trace(go.Scatter(x=d["x"][m], y=d["y"][m], mode="markers", hoverinfo="skip",
                                     name=f"margin {side} {depth:.2f} @ s {s0:g}-{s1:g}",
                                     marker=dict(symbol="circle-open", size=10, line_width=1.5,
                                                 color="#d62728" if side == "L" else "#1f77b4")), row=1, col=1)
    lap = f", {d['lap']:.2f} s on paper" if has_v else ""
    return _finish(fig, f"{TRACK} / {d['name']}: {d['length']:.2f} m{lap}")


CURSOR_JS = """
(function () {
  var gd = document.getElementById('{plot_id}');
  var cur = gd.data.length - 1;                      // the cursor trace, added last
  gd.on('plotly_hover', function (ev) {
    var p = ev.points[0];
    if (!p || !p.customdata) return;
    var s = p.customdata[0], x = p.customdata[1], y = p.customdata[2];
    Plotly.relayout(gd, {'shapes[0].x0': s, 'shapes[0].x1': s, 'shapes[0].visible': true,
                         'shapes[1].x0': s, 'shapes[1].x1': s, 'shapes[1].visible': true});
    Plotly.restyle(gd, {x: [[x]], y: [[y]]}, [cur]);
  });
})();
"""


def write_page(fig, path):
    """The figure as a standalone web page with the cursor linked across the three panels."""
    path = Path(path)
    fig.write_html(path, include_plotlyjs=True if EMBED_PLOTLY_JS else "cdn", post_script=CURSOR_JS)
    print(f"wrote {path}  ({path.stat().st_size / 1e6:.1f} MB)")
    return path

In [ ]:
fig = make_figure(d, tm, reg=frames.TRACKS.get(TRACK))
fig.show()
write_page(fig, LINE_DIR / f"view_{d['name']}.html")

## 3. Compare lines

Several lines on one map with their profiles overlaid. Click a legend entry to
hide that line in all three panels. The default set is the ICRA lineage: run
17, run 20 and run 23.

In [ ]:
LINES = [
    "raceline_a7.0zv_hard.csv",                 # run 17: 12.00 / 12.05, the flag-free fallback
    "raceline_a7.0zv_hard_l5.5.csv",            # run 20: 11.80 / 11.87, first longitudinal rung
    "raceline_a7.0zv_hard_l6.0_corners_h.csv",  # run 23: 11.60 / 11.66, the submission
]
COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b"]


def compare(names, tm):
    ds = [load_line(LINE_DIR / n, tm) for n in names]
    fig = _frame(("", "speed", "lateral demand v²|κ|"))
    fig.add_trace(map_trace(tm), row=1, col=1)
    for d, c in zip(ds, COLORS):
        cd, has_v = hover_data(d), d["v"] is not None
        tail = "<extra>" + d["name"] + "</extra>"
        fig.add_trace(go.Scatter(x=d["x"], y=d["y"], mode="lines", name=d["name"], legendgroup=d["name"],
                                 customdata=cd, line=dict(color=c, width=2),
                                 hovertemplate=(HOVER_LINE if has_v else HOVER_GEOM) + tail), row=1, col=1)
        if has_v:
            fig.add_trace(go.Scatter(x=d["s"], y=d["v"], mode="lines", name=d["name"], legendgroup=d["name"],
                                     showlegend=False, customdata=cd, line=dict(color=c),
                                     hovertemplate="s %{x:.2f} m: v %{y:.2f} m/s" + tail), row=1, col=2)
            fig.add_trace(go.Scatter(x=d["s"], y=d["a_lat"], mode="lines", name=d["name"], legendgroup=d["name"],
                                     showlegend=False, customdata=cd, line=dict(color=c),
                                     hovertemplate="s %{x:.2f} m: %{y:.2f} m/s²" + tail), row=2, col=2)
    return _finish(fig, f"{TRACK}: {len(ds)} lines"), ds


fig, ds = compare(LINES, tm)
for d in ds:
    describe(d)
fig.show()
write_page(fig, LINE_DIR / "view_compare.html")